# Heston-Arb — DoltHub free SPY data, 3-month gap-decomposition

**Goal:** run the repo's `gap_decomposition` diagnostic on FREE SPY option data
(DoltHub `post-no-preference/options`) over the last ~3 months, calibrating Heston
per session with prior-day Tikhonov regularization, and produce a per-day results table.

**Why this data:** no WRDS/OptionMetrics access; CBOE DataShop/ORATS cost money. DoltHub is
free, has real **bid/ask**, and updates ~daily. So unlike the Alpaca-based LEG B (trade closes,
no spreads), here we can also read the **round-trip spread cost** — the make-or-break term.

**Hard caveats baked into the data (do not over-read):**
- **SPY only** (SPX cash index is not in the dataset).
- **No open interest** → calibration weights are uniform (not OI/spread). Stated, not hidden.
- **No underlying spot** → derived per-expiry via put-call parity. Validated vs reality (~0.8% off).
- **Sparse, near-the-money, ~4 expiries/day, 11-60 day tenors** → per-day smile is fittable,
  but term-structure params (kappa, theta) are weakly identified on this span.

**What it can conclude:** `gap_decomposition` can *kill* the thesis (if gaps close by the model
chasing the market). It is necessary, not sufficient — a full net-of-cost P&L is the next step.
Run cells top to bottom. Needs only a GPU runtime + `GITHUB_TOKEN` in Colab secrets (no Alpaca/FRED keys).

In [ ]:
# ── Cell 1: GPU check ──────────────────────────────────────────────────────
import subprocess
r = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                   capture_output=True, text=True)
print('GPU:', r.stdout.strip() or 'No GPU — Runtime > Change runtime type > T4 (calibration runs on GPU)')

In [ ]:
# ── Cell 2: Dependencies (~2-3 min first run) ───────────────────────────────
# cvxpy / alpaca-py NOT needed: DoltHub + gap_decomposition never touch the LP repair path.
!pip install -q optax diffrax scipy pandas
!pip install -q -U "jax[cuda12]" -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html
import jax
print('JAX devices:', jax.devices())

In [ ]:
# ── Cell 3: Clone / update the repo ─────────────────────────────────────
# Requires GITHUB_TOKEN in Colab Secrets (key icon, left sidebar).
import os, subprocess
from google.colab import userdata

REPO = "/content/heston-arb"
REPO_URL = "github.com/AliAlpOezer/heston-arb.git"   # adjust if your remote differs
token = userdata.get("GITHUB_TOKEN")

if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", f"https://{token}@{REPO_URL}", REPO], check=True)
    print("Cloned.")
else:
    subprocess.run(["git", "-C", REPO, "pull"], check=True)
    print("Pulled latest.")
print("Files:", sorted(os.listdir(REPO)))

In [ ]:
# ── Cell 4: Imports + conventions ───────────────────────────────────────
import jax
jax.config.update("jax_enable_x64", True)   # branch-cut stability — MUST precede heston import

import sys, os
import numpy as np
import pandas as pd

REPO = "/content/heston-arb"
if REPO not in sys.path:
    sys.path.insert(0, REPO)
os.chdir(REPO)

import config
from calibration.heston import HestonParams
from data.dolthub import DoltHubLoader
from backtest.gap_decomposition import (
    _calibrate_session, build_pairs, summarize, report, _DEFAULT_INIT,
)

print("JAX devices:", jax.devices())
print(f"Gates | SIGNAL_MAX_RMSE={config.SIGNAL_MAX_RMSE}  MIN_VOL_GAP={config.MIN_VOL_GAP}  "
      f"MAX_SIGNAL_LM={config.MAX_SIGNAL_LOG_MONEYNESS}  MAX_CAL_LM={config.MAX_CAL_LOG_MONEYNESS}")

## Run: 63 SPY sessions → calibrate (prior-day regularized) → gap-decomposition

`_calibrate_session` restricts the fit to `|log-moneyness| <= MAX_CAL_LOG_MONEYNESS` (so a single-factor
Heston can fit it) and uses uniform weights (OI unavailable). We warm-start each day from the previous
fit and pass it as the Tikhonov anchor — daily data makes that prior meaningful.

In [ ]:
# ── Cell 5: Driver loop ──────────────────────────────────────────────
START, END   = "2026-03-24", "2026-06-23"   # last ~3 months; widen later
N_CAL_STEPS  = 300                           # Adam steps per session

loader = DoltHubLoader(call_only=False)      # both calls+puts -> larger calibration set
days = loader.candidate_days(START, END)     # business days; missing dates skipped below

sessions, per_day = [], []
init, prev = _DEFAULT_INIT, None
for day in days:
    try:
        chain = loader.fetch("SPY", day)     # ValueError on holidays / unrecorded dates
    except ValueError:
        continue
    s = _calibrate_session(chain, init, prev, N_CAL_STEPS)
    if s is None:                            # too few valid IVs to fit
        continue
    sessions.append(s)
    p = s["params"]
    # Real-spread cost context (only possible because DoltHub has bid/ask): median
    # round-trip relative spread (ask-bid)/mid for near-ATM contracts on this day.
    rel_spread = (chain.ask_prices - chain.bid_prices) / np.maximum(chain.mid_prices, 1e-6)
    atm = np.abs(chain.log_moneyness()) < 0.05
    atm_spread = float(np.median(rel_spread[atm])) if atm.any() else np.nan
    per_day.append(dict(
        date=s["date"], spot=round(chain.spot, 2), n_contracts=len(s["contracts"]),
        rmse=round(s["rmse"], 4), feller=bool(s["feller"]),
        kappa=round(float(p.kappa), 4), theta=round(float(p.theta), 5),
        xi=round(float(p.xi), 4), rho=round(float(p.rho), 4), v0=round(float(p.v0), 5),
        atm_rel_spread=round(atm_spread, 4),
    ))
    prev, init = s["params"], s["params"]    # warm-start + Tikhonov anchor
    print(f"{s['date']}: spot={chain.spot:7.2f}  n={len(s['contracts']):3d}  "
          f"rmse={s['rmse']:.4f}  feller={bool(s['feller'])}  atm_spread={atm_spread:.3f}")

print(f"\ncalibrated {len(sessions)} sessions out of {len(days)} business days")

In [ ]:
# ── Cell 6: Per-day results table ─────────────────────────────────────
df_days = pd.DataFrame(per_day)
print(f"sessions: {len(df_days)}   "
      f"fits passing RMSE gate (<= {config.SIGNAL_MAX_RMSE}): {(df_days.rmse <= config.SIGNAL_MAX_RMSE).sum()}   "
      f"Feller-OK: {df_days.feller.sum()}")
print(f"median ATM round-trip spread: {df_days.atm_rel_spread.median():.1%} of price")
df_days

In [ ]:
# ── Cell 7: Gap-decomposition diagnostic ─────────────────────────────────
rows = build_pairs(sessions)
summ = summarize(rows)
print(report(summ, f"{START}..{END}  (DoltHub free EOD, real bid/ask)"))

In [ ]:
# ── Cell 8: Cost sanity — does per-day convergence even cover the spread? ──────────
# market_contrib is vol-points EARNED per day; the option round-trip spread is paid ONCE.
# A position must hold long enough that (market_contrib/day x holding_days) > spread-in-vol-points.
# Here we give the gross convergence/day vs a crude spread-in-vol proxy for context only.
basis = "signal" if summ["n_signals"] > 0 else "gap_point"
mc = summ[basis]["market"]
print(f"basis={basis}: market_contrib mean = {mc['mean']*100:+.3f} vol-pts/day "
      f"(95% CI [{mc['lo']*100:+.3f}, {mc['hi']*100:+.3f}])")
print(f"median ATM round-trip price spread = {df_days.atm_rel_spread.median():.1%} of option price")
print("\nRead: if market_contrib/day <= 0 or its CI spans 0, the gap closes by the MODEL chasing the")
print("market -> no realized edge, and the spread cost is moot. A full net-of-cost P&L (vega-weighted")
print("spread vs realized convergence over the actual holding period) is the next step once this survives.")

In [ ]:
# ── Cell 9: Plots ────────────────────────────────────────────────
import matplotlib.pyplot as plt
d = pd.to_datetime(df_days.date)
fig, ax = plt.subplots(2, 2, figsize=(13, 8))
ax[0,0].plot(d, df_days.rmse, marker='.'); ax[0,0].axhline(config.SIGNAL_MAX_RMSE, color='r', ls='--', lw=0.8)
ax[0,0].set_title('calibration log-IV RMSE (red = signal gate)'); ax[0,0].tick_params(axis='x', rotation=45)
ax[0,1].plot(d, df_days.spot, marker='.', color='tab:blue'); ax[0,1].set_title('SPY spot (parity-derived)')
ax[0,1].tick_params(axis='x', rotation=45)
ax[1,0].plot(d, df_days.v0, label='v0'); ax[1,0].plot(d, df_days.theta, label='theta')
ax[1,0].set_title('variance params over time'); ax[1,0].legend(); ax[1,0].tick_params(axis='x', rotation=45)
ax[1,1].plot(d, df_days.atm_rel_spread, marker='.', color='tab:purple')
ax[1,1].set_title('ATM round-trip spread (frac of price)'); ax[1,1].tick_params(axis='x', rotation=45)
plt.tight_layout(); plt.show()

## How to read it

| Outcome | Meaning |
|---|---|
| **KILL-CONSISTENT** (market_contrib CI spans 0 or <= 0) | Gaps close by the **model chasing the market**. A held vol position earns nothing. Strategy structurally dead on this span. |
| **SURVIVES** (market_contrib CI > 0) | Real gaps close in the tradeable direction. **Necessary, not sufficient** — must still clear the spread cost and the gamma/realized-vol term this test ignores. |

**Limits of THIS run:** 3 months, ~63 sessions, SPY only, 4 expiries/day (weak kappa/theta), uniform
weights (no OI), parity-derived spot. It can falsify the edge cheaply; it cannot prove a durable
net-of-cost edge. If it survives, widen `START`/`END` (DoltHub goes back to ~2019) and build the
vega-weighted net-of-cost P&L using the real bid/ask we now have.